In [1]:
import pubmed_parser as pp

dicts_out = pp.parse_medline_xml('../data/pubmed_data/raw_data/44_abstracts_test.xml',
                                 year_info_only=False,
                                 nlm_category=False,
                                 author_list=False,
                                 reference_list=False)

In [2]:
import os

files = os.listdir("/home/rgoli/MetaMap-src/data/gold_standard/AnnotationResults_Abstracts/done")
print(files)

manual_pmids = [file.split('.')[0][4:] for file in files if file!='.ipynb_checkpoints']
print(manual_pmids)

['ABS_11604796.txt.done', 'ABS5_20841799.txt.done', 'ABS_29295283.txt.done', 'ABS_30306934.txt.done', 'ABS_28361157.txt.done', 'ABS_30871678.txt.done', 'ABS_29076113.txt.done', 'ABS5_22052898.txt.done', 'ABS_29433958.txt.done', 'ABS_19448882.txt.done', 'ABS_14664046.txt.done', 'ABS_24197356.txt.done', 'ABS_20501347.txt.done', 'ABS_29169646.txt.done', 'ABS_23304299.txt.done', 'ABS_29461493.txt.done', 'ABS_26133480.txt.done', 'ABS_18002176.txt.done', 'ABS_11198187.txt.done', 'ABS_18779083.txt.done', 'ABS_11418542.txt.done', 'ABS_26806717.txt.done', 'ABS5_12087117.txt.done', 'ABS_27401856.txt.done', 'ABS5_27301749.txt.done', 'ABS_28676255.txt.done', 'ABS_26539547.txt.done', 'ABS_22269224.txt.done', 'ABS_31438026.txt.done', 'ABS_31438019.txt.done', 'ABS_15360885.txt.done', 'ABS_17238374.txt.done', 'ABS_22326287.txt.done', 'ABS_23232759.txt.done', 'ABS_26590980.txt.done', 'ABS_22019377.txt.done', 'ABS5_23828174.txt.done', 'ABS_25109270.txt.done', 'ABS_26474836.txt.done', 'ABS_23523876.txt.d

In [3]:
pmid_manual_kw={}
for file in files:
    if file=='.ipynb_checkpoints': continue
    with open("/home/rgoli/MetaMap-src/data/gold_standard/AnnotationResults_Abstracts/done/"+file,'r') as fp:
        temp = fp.read().splitlines()
        pmid_manual_kw[temp[0].strip()]=temp[1:]

print(pmid_manual_kw['11604796'])

['drug-ordering system', 'optimizing', 'guideline-based ordering system ', 'enable', 'prescription errors ', 'operates', 'background process ', 'triggered', 'rules ', 'directs', 'browsing', 'development', 'clinical practice guidelines', 'developed', 'disseminate', 'design', 'decision tree']


In [4]:
import spacy
import scispacy

from scispacy.linking import EntityLinker

nlp = spacy.load("en_core_sci_lg")

2022-02-10 12:13:11.951329: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


In [5]:
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfTransformer from version 0.20.3 when using version 1.0.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfVectorizer from version 0.20.3 when using version 1.0.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [9]:
# tit_arr = []
# abs_arr = []
# pmid_arr = []
# for paper in dicts_out:
#     tit_arr.append(paper['title'])
#     abs_arr.append(paper['abstract'])
#     pmid_arr.append(paper['pmid'])
    
# print(tit_arr[0:2])
# print(abs_arr[0:2])
# print(pmid_arr[0:2])

In [10]:
# tit_kw = []
# for doc in nlp.pipe(tit_arr):
#     tit_kw.append(doc.ents)
    
# abs_kw = []
# for doc in nlp.pipe(abs_arr):
#     abs_kw.append(doc.ents)
    
# comb_kw = []
# i=0
# while i<len(abs_kw):
#     comb_kw.append(Union(abs_kw[i], tit_kw[i]))
#     i+=1

In [11]:
# print(tit_arr[0],"\n")
# print(abs_arr[0],"\n")
# print(list(set().union(abs_kw[0], tit_kw[0])))

In [12]:
comb_arr=[]
pmid_arr =[]
for paper in dicts_out:
    comb_arr.append(paper['title']+' '+paper['abstract'])
    pmid_arr.append(paper['pmid'])
    
print(pmid_arr[0:2])

['11604796', '24730353']


In [96]:
import re

comb_kw=[]
for doc in nlp.pipe(comb_arr[:10]):
    text = [token.orth_ for token in doc if not token.is_punct | token.is_space] 
    text_len = len(text)
#     print(text,"\n","-"*50)
    
    kws = [str(x) for x in doc.ents]
    kws.sort(reverse=True, key=len)
    kws = [x.split() for x in kws]
#     print(kws,"\n","-"*50)
    
    mapper = [0]*text_len
    result = []
    for kw in kws:
        kw_len=len(kw)
        i=0
        while i<text_len-kw_len:
            if text[i:i+kw_len]==kw and mapper[i:i+kw_len]==[0]*kw_len:
                mapper[i:i+kw_len]=[1]*kw_len
                result.append([kw,[i,i+kw_len]])
            i+=1
            
#     print(mapper,"\n","-"*50)
    for res in result:
        print("Position: {}\t\tWord: {}".format(res[1],' '.join(res[0])))
    
#     i=0
#     while i<text_len:
#         print(mapper[i],text[i])
# #         if mapper[i]==1:
# #             print("\033[91m {}".format(text[i]),sep=' '),
# #         else:
# #             print(text[i],sep=' '),
#         i+=1
    
#     result = []
#     for kw in kws:
#         kw_len = len(kw)
#         if kw_len==1 and kw[0] in text:
#             local_position = text.index(kw[0])
#             result.append([text.pop(local_position),local_position])
        
#         i=0
#         while i<text_len:
#             if kw
#             i+=1
    
#     print("="*100)
#     print(len(result),"/",len(kws))
#     print(result)
            
#     text=str(doc)
   
#     position =[]
#     for kw in doc.ents:
#         kw=str(kw)
#         first_occurance = text.find(kw)
#         last_position = position
#         position = [(m.start(0), m.end(0)) for m in re.finditer(kw,text,re.IGNORECASE)]
#         if len(position)>1 and len(last_position)==1:
#             for cpos in position:
#                 if cpos[0] > last_position[0][1]:
#                     position = [cpos]
#                     break
#                 else:
#                     position = [cpos]
#         elif len(position)>1 and len(last_position)==0:
#             position = [(first_occurance, first_occurance+len(kw))]
#         elif len(position)==0:
#             position =[(first_occurance, first_occurance+len(kw))]
#         print("FirstOccurance: ",first_occurance,'\tPosition: ',position,"\tWord: ",kw,"\Tag: ",doc[position[0][0]].tag_,"\tPOS: ",doc[position[0][0]].pos_)
    print("*"*150)
        
#     comb_kw.append(set(map(str,doc.ents)))

Position: [2, 5]		Word: guideline-based drug-ordering system
Position: [26, 29]		Word: patient-centered treatment strategy
Position: [59, 62]		Word: guideline-based ordering system
Position: [30, 33]		Word: clinical practice guidelines
Position: [38, 40]		Word: state-of-the-art information
Position: [64, 66]		Word: general practitioners
Position: [76, 78]		Word: therapeutic practices
Position: [16, 18]		Word: drug-centered checks
Position: [41, 43]		Word: treatment strategy
Position: [105, 106]		Word: recommendations
Position: [9, 10]		Word: computer-based
Position: [96, 97]		Word: automatically
Position: [119, 120]		Word: comprehensive
Position: [126, 128]		Word: decision tree
Position: [6, 8]		Word: primary care
Position: [68, 69]		Word: prescription
Position: [91, 92]		Word: prescription
Position: [133, 134]		Word: hypertension
Position: [37, 38]		Word: disseminate
Position: [79, 81]		Word: critic mode
Position: [107, 109]		Word: guided mode
Position: [13, 14]		Word: physicians
Posi

In [94]:
print(comb_arr[0],"\n")
print(comb_kw[0])

ASTI: a guideline-based drug-ordering system for primary care. Existing computer-based ordering systems for physicians provide effective drug-centered checks but offer little assistance for optimizing the overall patient-centered treatment strategy. Evidence-based clinical practice guidelines have been developed to disseminate state-of-the-art information concerning treatment strategy but these guidelines are poorly used in routine practice. The ASTI project aims to design a guideline-based ordering system to enable general practitioners to avoid prescription errors and to improve compliance with best therapeutic practices. The " critic mode " operates as a background process and corrects the physician's prescription on the basis of automatically triggered elementary rules that account for isolated guideline recommendations. The " guided mode " directs the physician to the best treatment by browsing a comprehensive guideline knowledge base represented as a decision tree. A first protot

IndexError: list index out of range

In [24]:
i=0
while i<len(comb_kw):
    print(pmid_arr[i],'\n')
    print(pmid_manual_kw[pmid_arr[i]],'\n')
    print(comb_kw[i],'\n')
    i+=1

11604796 

['drug-ordering system', 'optimizing', 'guideline-based ordering system ', 'enable', 'prescription errors ', 'operates', 'background process ', 'triggered', 'rules ', 'directs', 'browsing', 'development', 'clinical practice guidelines', 'developed', 'disseminate', 'design', 'decision tree'] 

{'hypertension', 'design', 'guidelines', 'prototype', 'project', 'prescription', 'guideline', 'development', 'guideline-based ordering system', 'errors', 'critic mode', 'comprehensive', 'browsing', 'automatically', 'general practitioners', 'guideline-based drug-ordering system', 'recommendations', 'practice', 'primary care', 'routine', 'systems', 'compliance', 'decision tree', 'disseminate', 'physician', 'isolated', 'patient-centered treatment strategy', 'therapeutic practices', 'ASTI', 'state-of-the-art information', 'clinical practice guidelines', 'treatment strategy', 'effective', 'assistance', "physician's", 'treatment', 'guided mode', 'physicians', 'drug-centered checks', 'computer

In [18]:
import pke

# define the valid Part-of-Speeches to occur in the graph
pos = {'NOUN', 'PROPN', 'ADJ'}

# define the grammar for selecting the keyphrase candidates
grammar = "NP: {<ADJ>*<NOUN|PROPN>+}"

# 1. create a PositionRank extractor.
extractor = pke.unsupervised.PositionRank()

# 2. load the content of the document.
extractor.load_document(input=comb_arr[0],
                        language='en',
                        normalization=None)

# 3. select the noun phrases up to 3 words as keyphrase candidates.
extractor.candidate_selection(grammar=grammar,
                              maximum_word_number=3)

# 4. weight the candidates using the sum of their word's scores that are
#    computed using random walk biaised with the position of the words
#    in the document. In the graph, nodes are words (nouns and
#    adjectives only) that are connected if they occur in a window of
#    10 words.
extractor.candidate_weighting(window=10,
                              pos=pos)

# 5. get the 10-highest scored candidates as keyphrases
keyphrases = extractor.get_n_best(n=20)
print(keyphrases)

[('guideline-based ordering system', 0.16362340378426526), ('guideline-based drug-ordering system', 0.144640632942711), ('computer-based ordering systems', 0.11636188582210048), ('asti project', 0.09891445693008956), ('asti', 0.08013096236746274), ('primary care', 0.06872959975231607), ('effective drug-centered checks', 0.06399992279098629), ('treatment strategy', 0.056314404346675916), ('routine practice', 0.046331426903449296), ('best treatment', 0.04506806652244158), ('prescription errors', 0.037805841754178185), ('little assistance', 0.037759580554456676), ('guidelines', 0.031011213689571604), ('best therapeutic practices', 0.02937246499729561), ('general practitioners', 0.02905019951753328), ('physicians', 0.025304414132300945), ('state-of-the-art information', 0.02391711270491774), ('prescription', 0.023339939826533467), ('critic mode', 0.018667874568557282), ('guideline recommendations', 0.014746059250420109)]


In [19]:
import pke
import string
from nltk.corpus import stopwords

# 1. create a MultipartiteRank extractor.
extractor = pke.unsupervised.MultipartiteRank()

# 2. load the content of the document.
extractor.load_document(input=comb_arr[0])

# 3. select the longest sequences of nouns and adjectives, that do
#    not contain punctuation marks or stopwords as candidates.
pos = {'NOUN', 'PROPN', 'ADJ'}
stoplist = list(string.punctuation)
stoplist += ['-lrb-', '-rrb-', '-lcb-', '-rcb-', '-lsb-', '-rsb-']
stoplist += stopwords.words('english')
extractor.candidate_selection(pos=pos, stoplist=stoplist)

# 4. build the Multipartite graph and rank candidates using random walk,
#    alpha controls the weight adjustment mechanism, see TopicRank for
#    threshold/method parameters.
extractor.candidate_weighting(alpha=1.1,
                              threshold=0.74,
                              method='average')

# 5. get the 10-highest scored candidates as keyphrases
keyphrases = extractor.get_n_best(n=20)
print(keyphrases)

[('guideline-based drug-ordering system', 0.09509511549054267), ('asti', 0.07823207588239302), ('physicians', 0.06729787535294751), ('overall patient-centered treatment strategy', 0.06474731710691091), ('primary care', 0.049771410291627266), ('prescription errors', 0.0406724827691195), ('guidelines', 0.036677989985406445), ('critic mode', 0.03389492033146792), ('effective drug-centered checks', 0.03329143995283239), ('evidence-based clinical practice guidelines', 0.03328420904854418), ('little assistance', 0.030121655733911186), ('computer-based ordering systems', 0.028968465997458507), ('state-of-the-art information', 0.02598239262328236), ('best therapeutic practices', 0.02553702950572291), ('compliance', 0.024809850439883135), ('general practitioners', 0.024356422881381682), ('treatment strategy', 0.024348658985516584), ('routine practice', 0.022846921353418512), ('prescription', 0.021787162723204228), ('guideline-based ordering system', 0.021199318684860427)]


In [20]:
print(pmid_manual_kw[pmid_arr[0]],'\n')

['drug-ordering system', 'optimizing', 'guideline-based ordering system ', 'enable', 'prescription errors ', 'operates', 'background process ', 'triggered', 'rules ', 'directs', 'browsing', 'development', 'clinical practice guidelines', 'developed', 'disseminate', 'design', 'decision tree'] 



In [23]:
import pke
import string
from nltk.corpus import stopwords

# 1. create a TopicRank extractor.
extractor = pke.unsupervised.TopicRank()

# 2. load the content of the document.
extractor.load_document(input=comb_arr[0])

# 3. select the longest sequences of nouns and adjectives, that do
#    not contain punctuation marks or stopwords as candidates.
pos = {'NOUN', 'PROPN', 'ADJ'}
stoplist = list(string.punctuation)
stoplist += ['-lrb-', '-rrb-', '-lcb-', '-rcb-', '-lsb-', '-rsb-']
stoplist += stopwords.words('english')
extractor.candidate_selection(pos=pos, stoplist=stoplist)

# 4. build topics by grouping candidates with HAC (average linkage,
#    threshold of 1/4 of shared stems). Weight the topics using random
#    walk, and select the first occuring candidate from each topic.
extractor.candidate_weighting(threshold=0.74, method='average')

# 5. get the 10-highest scored candidates as keyphrases
keyphrases = extractor.get_n_best(n=20)
print(keyphrases)

[('physicians', 0.0888403873735775), ('overall patient-centered treatment strategy', 0.08482354851434194), ('guideline-based drug-ordering system', 0.08100300358458583), ('prescription errors', 0.06326936466670863), ('guidelines', 0.05311235348860716), ('critic mode', 0.05247133728391051), ('asti', 0.04709528519972061), ('best therapeutic practices', 0.03618994629923039), ('effective drug-centered checks', 0.03581068598503181), ('compliance', 0.03502353958684092), ('basis', 0.03322368033962725), ('general practitioners', 0.03309114357952791), ('state-of-the-art information', 0.03242895254094424), ('evidence-based clinical practice guidelines', 0.031929811639632405), ('primary care', 0.03192936666229304), ('background process', 0.031223064785168078), ('routine practice', 0.0307813014512181), ('elementary rules', 0.03045161004750392), ('little assistance', 0.03044465824785106), ('decision tree', 0.030212655219831457)]
